In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ============================================================
# Federated SimCLR (ResNet-18) on PCam
# 1 split x 3 seeds (n=3), label_frac = 1%, 5%, 10%
# Outputs:
#   /kaggle/working/results/results_raw_resnet_federated_simclr.csv
#   /kaggle/working/results/results_summary_resnet_federated_simclr.csv
#   /kaggle/working/results/results_summary_resnet_federated_simclr.tex
# ============================================================

# -----------------------
# 0) List inputs
# -----------------------
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames[:5]:
        pass
print("Input root:", "/kaggle/input")
for dirname, _, filenames in os.walk('/kaggle/input'):
    if len(filenames) > 0:
        print(dirname, "->", len(filenames), "files")
        break

# -----------------------
# 1) Imports
# -----------------------
import json, time, math, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

import torchvision
from torchvision import transforms
from PIL import Image

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

# -----------------------
# 2) Config
# -----------------------
SPLIT_ID = 0
SEEDS = [0, 1, 2]
LABEL_FRACS = [0.01, 0.05, 0.10]

# Federated settings (simulated)
N_CLIENTS = 5
N_ROUNDS = 5
LOCAL_EPOCHS = 1

# SimCLR settings
SSL_EPOCHS_PER_LOCAL = 1  # local_epochs already controls; keep 1 here
SSL_BATCH = 128
SSL_LR = 1e-3
TEMPERATURE = 0.2

# Linear probe settings
PROBE_EPOCHS = 10
PROBE_BATCH = 256
PROBE_LR = 1e-3

OUT_DIR = "/kaggle/working/results"
os.makedirs(OUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# -----------------------
# 3) Utilities
# -----------------------
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def append_row_csv(path, row: dict):
    df = pd.DataFrame([row])
    if os.path.exists(path):
        df.to_csv(path, mode="a", header=False, index=False)
    else:
        df.to_csv(path, index=False)

def summarize_mean_std(raw_csv_path, group_cols, metric_cols, out_csv_path, out_tex_path=None):
    df = pd.read_csv(raw_csv_path)
    g = df.groupby(group_cols)[metric_cols].agg(["mean","std","count"]).reset_index()
    g.columns = ["_".join([c for c in col if c]) for col in g.columns.values]
    g.to_csv(out_csv_path, index=False)
    if out_tex_path:
        g.to_latex(out_tex_path, index=False, float_format="%.4f")
    return g

# Calibration metrics
def expected_calibration_error(probs, labels, n_bins=15):
    probs = np.asarray(probs)
    labels = np.asarray(labels).astype(int)
    confidences = np.maximum(probs, 1 - probs)
    preds = (probs >= 0.5).astype(int)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        mask = (confidences > lo) & (confidences <= hi)
        if mask.sum() == 0:
            continue
        acc_bin = (preds[mask] == labels[mask]).mean()
        conf_bin = confidences[mask].mean()
        ece += (mask.sum() / len(labels)) * abs(acc_bin - conf_bin)
    return float(ece)

def brier_score(probs, labels):
    probs = np.asarray(probs)
    labels = np.asarray(labels).astype(float)
    return float(np.mean((probs - labels) ** 2))

# -----------------------
# 4) Find PCam paths
# -----------------------
def find_pcam_root():
    # Try common Kaggle dataset folder names
    candidates = []
    for d in os.listdir("/kaggle/input"):
        candidates.append(os.path.join("/kaggle/input", d))
    # Look for train_labels.csv + train/
    for root in candidates:
        if os.path.exists(os.path.join(root, "train_labels.csv")) and os.path.isdir(os.path.join(root, "train")):
            return root
    # fallback: deep search
    for root, dirs, files in os.walk("/kaggle/input"):
        if "train_labels.csv" in files and "train" in dirs:
            return root
    raise FileNotFoundError("PCam dataset not found. Make sure you added Histopathologic Cancer Detection dataset.")

PCAM_ROOT = find_pcam_root()
print("PCAM_ROOT:", PCAM_ROOT)

train_csv = os.path.join(PCAM_ROOT, "train_labels.csv")
train_dir = os.path.join(PCAM_ROOT, "train")

labels_df = pd.read_csv(train_csv)
# Expect columns: id, label
assert "id" in labels_df.columns and "label" in labels_df.columns
labels_df["path"] = labels_df["id"].apply(lambda x: os.path.join(train_dir, f"{x}.tif"))
# Some Kaggle versions use .png, check existence
if not os.path.exists(labels_df["path"].iloc[0]):
    labels_df["path"] = labels_df["id"].apply(lambda x: os.path.join(train_dir, f"{x}.png"))

# Filter to existing files
exists_mask = labels_df["path"].apply(os.path.exists)
labels_df = labels_df[exists_mask].reset_index(drop=True)
print("PCam samples:", len(labels_df), "pos rate:", labels_df["label"].mean())

paths = labels_df["path"].values
y_all = labels_df["label"].values.astype(int)

# -----------------------
# 5) Split: train / val / test using StratifiedKFold
# -----------------------
def make_split_indices(y, n_splits=3, split_id=0, seed=0):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    splits = list(skf.split(np.zeros(len(y)), y))
    trainval_idx, test_idx = splits[split_id]

    # Further split trainval into train/val
    y_trainval = y[trainval_idx]
    skf2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed + 123)
    (train_idx_rel, val_idx_rel) = list(skf2.split(np.zeros(len(y_trainval)), y_trainval))[0]
    train_idx = trainval_idx[train_idx_rel]
    val_idx = trainval_idx[val_idx_rel]
    return train_idx, val_idx, test_idx

# We keep split fixed across seeds for fairness
train_idx0, val_idx0, test_idx0 = make_split_indices(y_all, n_splits=3, split_id=SPLIT_ID, seed=42)
print("Split sizes:", len(train_idx0), len(val_idx0), len(test_idx0))

# -----------------------
# 6) Datasets: SSL (two views) and Supervised
# -----------------------
class PCamImageDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        y = int(self.labels[i])
        return img, y

class PCamSSLDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        x1 = self.transform(img)
        x2 = self.transform(img)
        # label not used in SSL, but keep for non-IID split if needed
        y = int(self.labels[i])
        return x1, x2, y

# Transforms
ssl_tf = transforms.Compose([
    transforms.RandomResizedCrop(96, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
])

sup_tf = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
])

# Base datasets on fixed split
ssl_base = PCamSSLDataset(paths[train_idx0], y_all[train_idx0], transform=ssl_tf)
sup_train_base = PCamImageDataset(paths[train_idx0], y_all[train_idx0], transform=sup_tf)
sup_val = PCamImageDataset(paths[val_idx0], y_all[val_idx0], transform=sup_tf)
sup_test = PCamImageDataset(paths[test_idx0], y_all[test_idx0], transform=sup_tf)

# -----------------------
# 7) SimCLR model + loss
# -----------------------
class ResNetSimCLR(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        backbone = torchvision.models.resnet18(weights=None)
        # remove FC
        self.encoder = nn.Sequential(*list(backbone.children())[:-1])  # -> [B,512,1,1]
        self.proj = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, out_dim),
        )

    def forward(self, x):
        h = self.encoder(x).flatten(1)
        z = self.proj(h)
        z = F.normalize(z, dim=1)
        return h, z

def nt_xent_loss(z1, z2, temperature=0.2):
    """
    z1, z2: [B, D] normalized
    """
    B = z1.size(0)
    z = torch.cat([z1, z2], dim=0)  # [2B, D]
    sim = torch.mm(z, z.t()) / temperature  # [2B,2B]

    # mask self similarity
    mask = torch.eye(2*B, device=z.device).bool()
    sim = sim.masked_fill(mask, -1e9)

    # positives: i <-> i+B
    pos = torch.cat([torch.diag(sim, B), torch.diag(sim, -B)], dim=0)  # [2B]

    # denominator: logsumexp over row
    loss = -pos + torch.logsumexp(sim, dim=1)
    return loss.mean()

def simclr_train_one_epoch(model, loader, optimizer, device):
    model.train()
    total = 0.0
    n = 0
    for x1, x2, _ in loader:
        x1 = x1.to(device, non_blocking=True)
        x2 = x2.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        _, z1 = model(x1)
        _, z2 = model(x2)
        loss = nt_xent_loss(z1, z2, temperature=TEMPERATURE)
        loss.backward()
        optimizer.step()
        total += float(loss.item()) * x1.size(0)
        n += x1.size(0)
    return total / max(n, 1)

def get_resnet18_simclr_model():
    return ResNetSimCLR(out_dim=128)

# -----------------------
# 8) Federated simulation helpers
# -----------------------
def fedavg_state_dict(state_dicts):
    avg = {}
    for k in state_dicts[0].keys():
        avg[k] = sum(sd[k] for sd in state_dicts) / len(state_dicts)
    return avg

def make_clients_indices(n, n_clients, seed=0, non_iid=False, labels=None):
    rng = np.random.RandomState(seed)
    idx = np.arange(n)

    if non_iid and labels is not None:
        # sort by label to create label-skew, then chunk
        order = np.argsort(labels)
        idx = idx[order]

    rng.shuffle(idx)
    return np.array_split(idx, n_clients)

def run_federated_simclr(
    ssl_dataset,
    seed,
    n_clients=5,
    n_rounds=5,
    local_epochs=1,
    batch_size=128,
    non_iid=True,
):
    # client splits on SSL dataset indices
    labels_for_noniid = np.array([ssl_dataset[i][2] for i in range(len(ssl_dataset))], dtype=int)
    client_splits = make_clients_indices(
        n=len(ssl_dataset),
        n_clients=n_clients,
        seed=seed,
        non_iid=non_iid,
        labels=labels_for_noniid
    )

    global_model = get_resnet18_simclr_model().to(device)

    for rnd in range(n_rounds):
        local_states = []
        for c, c_idx in enumerate(client_splits):
            local_model = get_resnet18_simclr_model().to(device)
            local_model.load_state_dict(global_model.state_dict())

            subset = Subset(ssl_dataset, c_idx.tolist())
            loader = DataLoader(subset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)

            optimizer = torch.optim.Adam(local_model.parameters(), lr=SSL_LR)

            for _ in range(local_epochs):
                loss = simclr_train_one_epoch(local_model, loader, optimizer, device)

            local_states.append({k: v.detach().cpu() for k, v in local_model.state_dict().items()})
            print(f"[Fed] round {rnd+1}/{n_rounds} client {c+1}/{n_clients} loss={loss:.4f}")

        # FedAvg
        avg_sd = fedavg_state_dict(local_states)
        global_model.load_state_dict(avg_sd, strict=True)

    return global_model

# -----------------------
# 9) Linear probe eval (encoder frozen)
# -----------------------
class LinearProbe(nn.Module):
    def __init__(self, in_dim=512):
        super().__init__()
        self.fc = nn.Linear(in_dim, 1)

    def forward(self, h):
        return self.fc(h).squeeze(1)

@torch.no_grad()
def extract_features(encoder_model, loader):
    encoder_model.eval()
    feats = []
    labels = []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        # forward through encoder only
        h = encoder_model.encoder(x).flatten(1)
        feats.append(h.cpu().numpy())
        labels.append(y.numpy())
    return np.concatenate(feats, 0), np.concatenate(labels, 0)

def eval_linear_probe(encoder_model, label_frac, seed):
    # build labeled subset from train indices
    rng = np.random.RandomState(seed + 999)
    y_train = y_all[train_idx0]
    idx_train = np.arange(len(train_idx0))

    # stratified subsample
    pos = idx_train[y_train == 1]
    neg = idx_train[y_train == 0]
    n_pos = max(1, int(len(pos) * label_frac))
    n_neg = max(1, int(len(neg) * label_frac))
    sub_pos = rng.choice(pos, size=n_pos, replace=False)
    sub_neg = rng.choice(neg, size=n_neg, replace=False)
    sub_idx_rel = np.concatenate([sub_pos, sub_neg])
    rng.shuffle(sub_idx_rel)

    sup_sub = Subset(sup_train_base, sub_idx_rel.tolist())
    train_loader = DataLoader(sup_sub, batch_size=PROBE_BATCH, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(sup_val, batch_size=PROBE_BATCH, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(sup_test, batch_size=PROBE_BATCH, shuffle=False, num_workers=2, pin_memory=True)

    # extract fixed features
    Xtr, ytr = extract_features(encoder_model, train_loader)
    Xva, yva = extract_features(encoder_model, val_loader)
    Xte, yte = extract_features(encoder_model, test_loader)

    # train linear probe on GPU (small MLP = linear)
    probe = LinearProbe(in_dim=Xtr.shape[1]).to(device)
    opt = torch.optim.Adam(probe.parameters(), lr=PROBE_LR)
    bce = nn.BCEWithLogitsLoss()

    Xtr_t = torch.tensor(Xtr, dtype=torch.float32, device=device)
    ytr_t = torch.tensor(ytr, dtype=torch.float32, device=device)
    Xva_t = torch.tensor(Xva, dtype=torch.float32, device=device)
    yva_t = torch.tensor(yva, dtype=torch.float32, device=device)

    best_state = None
    best_val = 1e9

    for ep in range(PROBE_EPOCHS):
        probe.train()
        # mini-batch train over features
        perm = torch.randperm(Xtr_t.size(0), device=device)
        total = 0.0
        n = 0
        for i in range(0, Xtr_t.size(0), PROBE_BATCH):
            idx = perm[i:i+PROBE_BATCH]
            logits = probe(Xtr_t[idx])
            loss = bce(logits, ytr_t[idx])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            total += float(loss.item()) * idx.numel()
            n += idx.numel()

        probe.eval()
        with torch.no_grad():
            val_logits = probe(Xva_t)
            val_loss = float(bce(val_logits, yva_t).item())
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu() for k, v in probe.state_dict().items()}

    # test
    probe.load_state_dict(best_state, strict=True)
    probe.eval()
    Xte_t = torch.tensor(Xte, dtype=torch.float32, device=device)
    with torch.no_grad():
        logits = probe(Xte_t).cpu().numpy()
        probs = 1 / (1 + np.exp(-logits))

    preds = (probs >= 0.5).astype(int)

    acc = accuracy_score(yte, preds)
    f1 = f1_score(yte, preds)
    try:
        auc = roc_auc_score(yte, probs)
    except Exception:
        auc = float("nan")
    ece = expected_calibration_error(probs, yte, n_bins=15)
    brier = brier_score(probs, yte)

    return {"acc": float(acc), "auc": float(auc), "f1": float(f1), "ece": float(ece), "brier": float(brier)}

# -----------------------
# 10) Run: 1 split x 3 seeds (n=3)
# -----------------------
RAW_PATH = f"{OUT_DIR}/results_raw_resnet_federated_simclr.csv"

for seed in SEEDS:
    set_all_seeds(seed)
    print("\n==============================")
    print("SEED =", seed)
    print("==============================")

    # Federated pretrain once per seed (then probe for each label fraction)
    t0 = time.time()
    fed_model = run_federated_simclr(
        ssl_dataset=ssl_base,
        seed=seed,
        n_clients=N_CLIENTS,
        n_rounds=N_ROUNDS,
        local_epochs=LOCAL_EPOCHS,
        batch_size=SSL_BATCH,
        non_iid=True
    )
    t1 = time.time()
    print(f"Federated pretrain done in {(t1-t0)/60:.2f} min")

    for frac in LABEL_FRACS:
        print("RUN probe", "split", SPLIT_ID, "seed", seed, "frac", frac)
        metrics = eval_linear_probe(fed_model, label_frac=frac, seed=seed)

        row = {
            "exp": "federated_simclr",
            "backbone": "resnet18",
            "dataset": "pcam",
            "split_id": SPLIT_ID,
            "seed": seed,
            "label_frac": frac,
            **metrics
        }
        append_row_csv(RAW_PATH, row)
        print("metrics:", metrics)

# -----------------------
# 11) Summarize mean±std
# -----------------------
summarize_mean_std(
    RAW_PATH,
    group_cols=["exp", "backbone", "dataset", "label_frac"],
    metric_cols=["acc", "auc", "f1", "ece", "brier"],
    out_csv_path=f"{OUT_DIR}/results_summary_resnet_federated_simclr.csv",
    out_tex_path=f"{OUT_DIR}/results_summary_resnet_federated_simclr.tex",
)

print("\nSaved:")
print("RAW:", RAW_PATH)
print("SUMMARY CSV:", f"{OUT_DIR}/results_summary_resnet_federated_simclr.csv")
print("SUMMARY TEX:", f"{OUT_DIR}/results_summary_resnet_federated_simclr.tex")
